In [10]:
!pip install pytesseract
!apt-get update -qq
!apt-get install -y tesseract-ocr

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (5.3.4-1build5).
0 upgraded, 0 newly installed, 0 to remove and 40 not upgraded.


In [16]:
"""
Project 4 - Path 1: Optical Character Recognition (OCR)
DecodeLabs Industrial Training Kit

Satisfies all 4 Gatekeeper Rule checkpoints:
  1. Library Integration      -> pytesseract, error-free
  2. Pre-Processing Integrity -> Grayscale + Adaptive Thresholding
  3. Accuracy Benchmarking    -> keeps only words with confidence >= 80%
  4. Visual Confirmation      -> prints clean text + saves annotated image
"""

import cv2
import pytesseract
from pytesseract import Output

# ---- CONFIG ----
IMAGE_PATH = "/content/sample_text (1).png"        # <-- change to your own image
CONFIDENCE_THRESHOLD = 80             # rubric minimum: 80%
OUTPUT_PATH = "ocr_output_annotated.png"




"""Reads the raw image, converts it from color (RGB) to grayscale to reduce data complexity,
then applies Gaussian blur to remove noise,
and finally Otsu's adaptive thresholding to turn it into pure black-and-white. This is the mandatory
"Pre-Processing Integrity" step — it prepares a clean, high-contrast image so the OCR engine can read characters accurately."""
def preprocess_image(image_path):

    original = cv2.imread(image_path)
    if original is None:
        raise FileNotFoundError(f"Could not read image at '{image_path}'")

    # Step 1: Grayscale Conversion - collapses 3D RGB matrix to 1D intensity
    gray = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)

    # Light Gaussian blur to remove noise before thresholding
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Adaptive Thresholding (Otsu's Method) - forces binary black/white
    _, thresh = cv2.threshold(
        blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    return original, thresh




In [17]:
"""
Sends the cleaned image into pytesseract (the Python wrapper for Google's Tesseract OCR engine).
It returns every detected word along with its bounding box position and a confidence score.
This is the core "Library Integration" step — it's what actually performs the recognition.
"""
def run_ocr(processed_image, psm=6):

    config = f"--psm {psm}"
    data = pytesseract.image_to_data(
        processed_image, config=config, output_type=Output.DICT
    )
    return data




In [18]:
"""
Loops through all detected words and keeps only the ones with confidence ≥ 80%, discarding the rest.
 This enforces the rubric's "Accuracy Benchmarking" requirement — it stops low-confidence guesses (noise, misreads) from polluting the final output.
"""
def filter_by_confidence(data, threshold=CONFIDENCE_THRESHOLD):

    kept_words = []
    boxes = []
    n = len(data["text"])
    for i in range(n):
        text = data["text"][i].strip()
        conf = int(float(data["conf"][i])) if data["conf"][i] != "-1" else -1
        if text and conf >= threshold:
            kept_words.append(text)
            boxes.append((
                data["left"][i], data["top"][i],
                data["width"][i], data["height"][i], conf
            ))
    return kept_words, boxes




In [19]:
"""Draws a green bounding box and confidence label around each word that passed the filter,
then saves the annotated image to disk.
 This satisfies "Visual Confirmation" — proof that the recognition actually worked, not just printed text."""
def draw_confirmation(original_image, boxes, output_path):

    annotated = original_image.copy()
    for (x, y, w, h, conf) in boxes:
        cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 200, 0), 2)
        cv2.putText(
            annotated, f"{conf}%", (x, max(0, y - 8)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 0), 2
        )
    cv2.imwrite(output_path, annotated)
    return output_path




In [20]:
def main():
    original, processed = preprocess_image(IMAGE_PATH)
    data = run_ocr(processed, psm=6)
    words, boxes = filter_by_confidence(data, CONFIDENCE_THRESHOLD)

    final_text = " ".join(words)
    print("=" * 50)
    print(f"VALIDATED OUTPUT (confidence >= {CONFIDENCE_THRESHOLD}%)")
    print("=" * 50)
    print(final_text if final_text else "[No text passed the confidence gate]")
    print("-" * 50)
    for w, (x, y, wd, ht, c) in zip(words, boxes):
        print(f"  '{w}'  -> confidence: {c}%  @ ({x},{y},{wd},{ht})")

    saved_path = draw_confirmation(original, boxes, OUTPUT_PATH)
    print(f"\nAnnotated confirmation image saved to: {saved_path}")


if __name__ == "__main__":
    main()

VALIDATED OUTPUT (confidence >= 80%)
Hello World 2026
--------------------------------------------------
  'Hello'  -> confidence: 94%  @ (34,67,115,32)
  'World'  -> confidence: 95%  @ (168,67,134,32)
  '2026'  -> confidence: 96%  @ (322,67,112,32)

Annotated confirmation image saved to: ocr_output_annotated.png
